In [49]:
import sqlite3
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langgraph.checkpoint.sqlite import SqliteSaver


llm = init_chat_model("openai:gpt-4o-mini")
conn = sqlite3.connect(
    "memory.db",
    check_same_thread = False, 
)

config = {
    "configurable":{
        "thread_id":"3",
    }
}


In [50]:
class State(MessagesState):
    pass 

graph_builder = StateGraph(State)


In [51]:
def chatbot(state:State):
    response = llm.invoke(state["messages"])
    return {
        "messages":[response]
    }

    

In [52]:
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot",END)

graph = graph_builder.compile(
    checkpointer = SqliteSaver(conn),
)

In [53]:
result = graph.invoke(
    {
        "messages":[
            {"role":"user","content":"I live in Europe now. And the city I live in is Paris."}
        ]
    },
    config = config,
)

In [54]:
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Hello my name is HS
================================== Ai Message ==================================

Hello HS! How can I assist you today?
================================ Human Message =================================

I live in Europe now. And the city I live in is Paris.
================================== Ai Message ==================================

That's great! Paris is a fantastic city with a rich history, amazing culture, and incredible cuisine. What do you enjoy most about living there?
================================ Human Message =================================

I live in Europe now. And the city I live in is Paris.
================================== Ai Message ==================================

That's wonderful! Paris is known for its beautiful architecture, art, and vibrant atmosphere. Are there any specific aspects of the city that you're particularly fond of? Or are there any places 

In [55]:
state_history = graph.get_state_history(config)
to_fork = list(state_history)[-5]
to_fork.config


{'configurable': {'thread_id': '3',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1235fa-60a4-6be3-8003-4e2ee995c688'}}

In [56]:
for state_snapshot in list(state_history):
    print(state_snapshot.next)
    print( state_snapshot.values["messages"])
    
    

In [ ]:
from langchain_core.messages import message

graph.update_state(
    to_fork.config,
    {
        "messages":Human
    }
)